In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import re

import numpy as np


_FLOAT_PATTERN = (
    r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[EeDd][-+]?\d+)?"
)
_FLOAT_RE = re.compile(_FLOAT_PATTERN)


@dataclass
class LCModelBasis:
    """
    Parsed LCModel BASIS file.

    Notes
    -----
    `raw_spectra` contains the complex data exactly as stored in the
    LCModel BASIS file. These arrays are not yet converted into the
    final time-domain basis FIDs used for simulation.
    """

    names: list[str]
    raw_spectra: np.ndarray

    dwell_time: float
    hz_per_ppm: float
    echo_time: float
    sequence: str

    ids: list[str]
    concentrations: np.ndarray
    tramps: np.ndarray
    volumes: np.ndarray
    ishifts: np.ndarray

    @property
    def n_metabolites(self) -> int:
        return self.raw_spectra.shape[0]

    @property
    def n_points(self) -> int:
        return self.raw_spectra.shape[1]

    @property
    def sampling_rate(self) -> float:
        return 1.0 / self.dwell_time

    @property
    def bandwidth(self) -> float:
        return self.sampling_rate

    @property
    def time_axis(self) -> np.ndarray:
        return np.arange(self.n_points) * self.dwell_time

    def as_dict(self) -> dict[str, np.ndarray]:
        return {
            name: self.raw_spectra[i]
            for i, name in enumerate(self.names)
        }


def _fortran_float(value: str) -> float:
    """Convert Fortran-style D/E floating point text to Python float."""
    return float(
        value.replace("D", "E").replace("d", "e")
    )


def _read_float(block: str, key: str) -> float:
    match = re.search(
        rf"\b{re.escape(key)}\s*=\s*({_FLOAT_PATTERN})",
        block,
        flags=re.IGNORECASE,
    )

    if match is None:
        raise ValueError(
            f"Could not find numeric field '{key}'."
        )

    return _fortran_float(match.group(1))


def _read_int(block: str, key: str) -> int:
    return int(round(_read_float(block, key)))


def _read_string(block: str, key: str) -> str:
    match = re.search(
        rf"\b{re.escape(key)}\s*=\s*'([^']*)'",
        block,
        flags=re.IGNORECASE,
    )

    if match is None:
        raise ValueError(
            f"Could not find string field '{key}'."
        )

    return match.group(1).strip()


def _find_header_block(
    text: str,
    block_name: str,
) -> str:
    match = re.search(
        rf"(?ms)^\s*\${re.escape(block_name)}\s*$"
        rf"\s*(.*?)"
        rf"^\s*\$END\s*$",
        text,
    )

    if match is None:
        raise ValueError(
            f"Could not find ${block_name} block."
        )

    return match.group(1)


def load_lcmodel_basis(
    path: str | Path,
    *,
    dtype: np.dtype = np.complex64,
) -> LCModelBasis:
    """
    Read an LCModel .basis file.

    Parameters
    ----------
    path:
        Path to the LCModel BASIS file.

    dtype:
        Complex dtype used for the stored raw spectra.

    Returns
    -------
    LCModelBasis
        Parsed raw basis spectra and associated metadata.
    """
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"Basis file does not exist: {path}"
        )

    text = path.read_text(
        encoding="latin-1",
        errors="strict",
    )

    seqpar_block = _find_header_block(
        text,
        "SEQPAR",
    )

    basis1_block = _find_header_block(
        text,
        "BASIS1",
    )

    hz_per_ppm = _read_float(
        seqpar_block,
        "HZPPPM",
    )

    echo_time = _read_float(
        seqpar_block,
        "ECHOT",
    )

    sequence = _read_string(
        seqpar_block,
        "SEQ",
    )

    dwell_time = _read_float(
        basis1_block,
        "BADELT",
    )

    n_points = _read_int(
        basis1_block,
        "NDATAB",
    )

    basis_block_pattern = re.compile(
        r"(?ms)"
        r"^\s*\$BASIS\s*$"
        r"\s*(.*?)"
        r"^\s*\$END\s*$"
    )

    basis_matches = list(
        basis_block_pattern.finditer(text)
    )

    if not basis_matches:
        raise ValueError(
            "No $BASIS metabolite blocks found."
        )

    names: list[str] = []
    ids: list[str] = []

    concentrations: list[float] = []
    tramps: list[float] = []
    volumes: list[float] = []
    ishifts: list[int] = []

    raw_spectra: list[np.ndarray] = []

    for i, match in enumerate(basis_matches):
        metadata_block = match.group(1)

        data_start = match.end()

        data_end = (
            basis_matches[i + 1].start()
            if i + 1 < len(basis_matches)
            else len(text)
        )

        data_block = text[data_start:data_end]

        next_nmused = re.search(
            r"(?m)^\s*\$NMUSED\s*$",
            data_block,
        )

        if next_nmused is not None:
            data_block = data_block[
                :next_nmused.start()
            ]

        numeric_strings = _FLOAT_RE.findall(
            data_block
        )

        values = np.fromiter(
            (
                _fortran_float(value)
                for value in numeric_strings
            ),
            dtype=np.float64,
        )

        expected_values = 2 * n_points

        if values.size != expected_values:
            metabolite_name = _read_string(
                metadata_block,
                "METABO",
            )

            raise ValueError(
                f"Unexpected number of values for "
                f"'{metabolite_name}': "
                f"found {values.size}, "
                f"expected {expected_values} "
                f"({n_points} complex points)."
            )

        raw_spectrum = (
            values[0::2]
            + 1j * values[1::2]
        )

        names.append(
            _read_string(
                metadata_block,
                "METABO",
            )
        )

        ids.append(
            _read_string(
                metadata_block,
                "ID",
            )
        )

        concentrations.append(
            _read_float(
                metadata_block,
                "CONC",
            )
        )

        tramps.append(
            _read_float(
                metadata_block,
                "TRAMP",
            )
        )

        volumes.append(
            _read_float(
                metadata_block,
                "VOLUME",
            )
        )

        ishifts.append(
            _read_int(
                metadata_block,
                "ISHIFT",
            )
        )

        raw_spectra.append(
            raw_spectrum.astype(
                dtype,
                copy=False,
            )
        )

    raw_spectra_array = np.stack(
        raw_spectra,
        axis=0,
    )

    return LCModelBasis(
        names=names,
        raw_spectra=raw_spectra_array,
        dwell_time=dwell_time,
        hz_per_ppm=hz_per_ppm,
        echo_time=echo_time,
        sequence=sequence,
        ids=ids,
        concentrations=np.asarray(
            concentrations,
            dtype=np.float32,
        ),
        tramps=np.asarray(
            tramps,
            dtype=np.float32,
        ),
        volumes=np.asarray(
            volumes,
            dtype=np.float32,
        ),
        ishifts=np.asarray(
            ishifts,
            dtype=np.int32,
        ),
    )

In [ ]:
basis = load_lcmodel_basis(
    "LCModelBasis/raw/7T.basis"
)

print(basis.names)
print(basis.raw_spectra.shape)
print(basis.dwell_time)
print(basis.sampling_rate)
print(basis.hz_per_ppm)

In [ ]:
naa_fid = basis.as_dict()["NAA"]
print(naa_fid.shape)

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# LCModel-Rohspektren in die übliche Darstellungsreihenfolge bringen
# (entspricht dem Vorgehen des FID-A LCModel-Readers)
# ------------------------------------------------------------
raw_spectra = basis.raw_spectra

spectra = np.flip(
    np.fft.fftshift(
        np.conj(raw_spectra),
        axes=-1,
    ),
    axis=-1,
)


# ------------------------------------------------------------
# Gemeinsame ppm-Achse
# ------------------------------------------------------------
n_points = spectra.shape[-1]
spectral_width_hz = basis.bandwidth

frequency_hz = np.linspace(
    -spectral_width_hz / 2
    + spectral_width_hz / (2 * n_points),
    spectral_width_hz / 2
    - spectral_width_hz / (2 * n_points),
    n_points,
)

ppm = (
    frequency_hz / basis.hz_per_ppm
    + 4.68
)


# ------------------------------------------------------------
# Nur interessanten MRS-Bereich darstellen
# ------------------------------------------------------------
plot_mask = (
    (ppm >= 0.0)
    & (ppm <= 7.5)
)


# ------------------------------------------------------------
# Rasterplot
# ------------------------------------------------------------
n_spectra = basis.n_metabolites
n_columns = 4
n_rows = math.ceil(n_spectra / n_columns)

fig, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(16, 3 * n_rows),
    sharex=True,
)

axes = np.asarray(axes).ravel()

for ax, name, spectrum in zip(
    axes,
    basis.names,
    spectra,
):
    ax.plot(
        ppm[plot_mask],
        np.abs(spectrum[plot_mask]),
    )

    ax.set_title(name)
    ax.set_xlim(7.0, 0.0)
    ax.grid(alpha=0.3)

for ax in axes[n_spectra:]:
    ax.axis("off")

fig.supxlabel("Chemical shift [ppm]")
fig.supylabel("Magnitude [a.u.]")
fig.suptitle("LCModel basis spectra")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np


def prepare_basis_fid_for_acquisition(
    native_spectrum: np.ndarray,
    *,
    source_dwell_time: float,
    target_bandwidth: float,
    target_n_timepoints: int,
):
    """
    Convert a native LCModel basis spectrum to the desired acquisition.

    The spectrum is cropped to the requested bandwidth and transformed
    into the time domain. Finally, the desired number of FID points is
    retained.

    Parameters
    ----------
    native_spectrum:
        Native LCModel basis spectrum in fftshift ordering.

    source_dwell_time:
        Native LCModel dwell time (BADELT).

    target_bandwidth:
        Desired acquisition bandwidth in Hz.

    target_n_timepoints:
        Desired number of acquired FID points.

    Returns
    -------
    target_fid:
        Cropped time-domain basis function.

    actual_bandwidth:
        Actual bandwidth after integer frequency cropping.
    """
    n_source = native_spectrum.size

    source_frequency_resolution = (
        1.0
        / (n_source * source_dwell_time)
    )

    n_cropped = int(
        round(
            target_bandwidth
            / source_frequency_resolution
        )
    )

    if n_cropped > n_source:
        raise ValueError(
            "Target bandwidth exceeds native basis bandwidth."
        )

    center = n_source // 2

    start = center - n_cropped // 2
    stop = start + n_cropped

    cropped_spectrum = native_spectrum[start:stop]

    actual_bandwidth = (
        n_cropped
        * source_frequency_resolution
    )

    full_fid = np.fft.ifft(
        np.fft.ifftshift(cropped_spectrum)
    )

    if target_n_timepoints > full_fid.size:
        raise ValueError(
            f"Requested {target_n_timepoints} time points, "
            f"but only {full_fid.size} are available."
        )

    target_fid = full_fid[:target_n_timepoints]

    return target_fid, actual_bandwidth

In [ ]:
import hlsvdpropy

print(hlsvdpropy.__version__)

In [ ]:
import numpy as np


def lcmodel_component_to_fid(
    basis: LCModelBasis,
    metabolite: str,
) -> np.ndarray:
    """
    Convert one raw LCModel BASIS component to a time-domain FID.

    Processing steps
    ----------------
    1. Select the requested metabolite.
    2. Apply the LCModel ISHIFT value.
    3. Transform to the time domain using an inverse FFT.
    4. Apply complex conjugation.
    """
    metabolite_index = basis.names.index(metabolite)

    raw_spectrum = np.asarray(
        basis.raw_spectra[metabolite_index],
        dtype=np.complex128,
    )

    ishift = int(
        basis.ishifts[metabolite_index]
    )

    shifted_spectrum = np.roll(
        raw_spectrum,
        -ishift,
    )

    fid = np.conj(
        np.fft.ifft(shifted_spectrum)
    )

    return fid

In [ ]:
def model_reference_peak_hlsvd(
    fid: np.ndarray,
    *,
    dwell_time: float,
    hz_per_ppm: float,
    ppm_limits: tuple[float, float] = (-0.2, 0.2),
    ppm_reference: float = 4.65,
    n_singular_values: int = 5,
    n_fit_points: int = 8192,
) -> tuple[np.ndarray, dict]:
    """
    Estimate the reference peak from an initial segment of the FID,
    then reconstruct it over the complete FID.
    """
    fid = np.asarray(fid, dtype=np.complex128)

    if n_fit_points > fid.size:
        raise ValueError(
            f"n_fit_points={n_fit_points} exceeds FID length={fid.size}."
        )

    # Only use a manageable initial segment for the expensive HLSVD fit
    fit_fid = fid[:n_fit_points]
    m = fit_fid.size // 2

    raw_result = hlsvdpropy.hlsvdpro(
        fit_fid,
        n_singular_values,
        m=m,
        sparse=True,
    )

    converted_result = hlsvdpropy.convert_hlsvd_result(
        raw_result,
        dwell_time,
    )

    (
        n_singular_values_found,
        singular_values,
        frequencies_hz,
        damping_times,
        amplitudes,
        phases_deg,
    ) = converted_result[:6]

    frequencies_hz = np.asarray(frequencies_hz)
    damping_times = np.asarray(damping_times)
    amplitudes = np.asarray(amplitudes)
    phases_deg = np.asarray(phases_deg)
    singular_values = np.asarray(singular_values)

    frequency_limits_hz = (
        np.asarray(ppm_limits) - ppm_reference
    ) * hz_per_ppm

    selected = (
        (frequencies_hz > frequency_limits_hz[0])
        & (frequencies_hz < frequency_limits_hz[1])
    )

    # Reconstruct the selected components over the FULL original FID
    full_time_axis = (
        np.arange(fid.size, dtype=np.float64)
        * dwell_time
    )

    reference_fid = np.zeros_like(
        fid,
        dtype=np.complex128,
    )

    for use, frequency, damping, amplitude, phase in zip(
        selected,
        frequencies_hz,
        damping_times,
        amplitudes,
        phases_deg,
    ):
        if use:
            reference_fid += amplitude * np.exp(
                full_time_axis / damping
                + 1j
                * 2.0
                * np.pi
                * (
                    frequency * full_time_axis
                    + phase / 360.0
                )
            )

    component_ppm = (
        frequencies_hz / hz_per_ppm
        + ppm_reference
    )

    info = {
        "n_fit_points": n_fit_points,
        "hankel_shape": (
            fit_fid.size - m,
            m + 1,
        ),
        "n_singular_values_found": n_singular_values_found,
        "singular_values": singular_values,
        "frequencies_hz": frequencies_hz,
        "component_ppm": component_ppm,
        "damping_times": damping_times,
        "amplitudes": amplitudes,
        "phases_deg": phases_deg,
        "selected": selected,
        "selected_frequencies_hz": frequencies_hz[selected],
        "selected_ppm": component_ppm[selected],
        "selected_damping_times": damping_times[selected],
        "selected_amplitudes": amplitudes[selected],
        "selected_phases_deg": phases_deg[selected],
    }

    return reference_fid, info

In [ ]:
tau_fid = lcmodel_component_to_fid(
    basis,
    "Tau",
)

tau_reference_fid, tau_hlsvd_info = (
    model_reference_peak_hlsvd(
        tau_fid,
        dwell_time=basis.dwell_time,
        hz_per_ppm=basis.hz_per_ppm,
        ppm_limits=(-0.2, 0.2),
        ppm_reference=4.65,
        n_singular_values=5,
        n_fit_points=8192,
    )
)

tau_clean_fid = tau_fid - tau_reference_fid

print(
    "Hankel matrix:",
    tau_hlsvd_info["hankel_shape"],
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# In den Frequenzraum
# ------------------------------------------------------------
tau_original_spectrum = np.fft.fftshift(
    np.fft.fft(tau_fid)
)

tau_clean_spectrum = np.fft.fftshift(
    np.fft.fft(tau_clean_fid)
)

tau_reference_spectrum = np.fft.fftshift(
    np.fft.fft(tau_reference_fid)
)


# ------------------------------------------------------------
# ppm-Achse (native Bandbreite)
# ------------------------------------------------------------
n_points = tau_fid.size
spectral_width_hz = 1.0 / basis.dwell_time

frequency_hz = np.linspace(
    -spectral_width_hz / 2 + spectral_width_hz / (2 * n_points),
    spectral_width_hz / 2 - spectral_width_hz / (2 * n_points),
    n_points,
)

ppm = frequency_hz / basis.hz_per_ppm + 4.65


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, axes = plt.subplots(
    2,
    1,
    figsize=(11, 8),
)

# Gesamter metabolischer Bereich
axes[0].plot(
    ppm,
    np.abs(tau_original_spectrum),
    label="Original",
)

axes[0].plot(
    ppm,
    np.abs(tau_clean_spectrum),
    "--",
    label="After HLSVD",
)

axes[0].plot(
    ppm,
    np.abs(tau_reference_spectrum),
    label="Removed reference",
    alpha=0.8,
)

axes[0].set_xlim(4.5, -0.5)
axes[0].set_xlabel("Chemical shift [ppm]")
axes[0].set_ylabel("Magnitude")
axes[0].set_title("Tau basis before / after HLSVD")
axes[0].grid(alpha=0.3)
axes[0].legend()


# Zoom auf Referenzpeak
axes[1].plot(
    ppm,
    np.abs(tau_original_spectrum),
    label="Original",
)

axes[1].plot(
    ppm,
    np.abs(tau_clean_spectrum),
    "--",
    label="After HLSVD",
)

axes[1].plot(
    ppm,
    np.abs(tau_reference_spectrum),
    label="Removed reference",
    alpha=0.8,
)

axes[1].set_xlim(0.5, -0.5)
axes[1].set_xlabel("Chemical shift [ppm]")
axes[1].set_ylabel("Magnitude")
axes[1].set_title("Zoom around 0 ppm")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# HLSVD reference removal for all basis components
# ------------------------------------------------------------
original_fids = []
clean_fids = []
reference_fids = []
hlsvd_info_by_metabolite = {}


for metabolite in basis.names:
    print(f"Processing {metabolite}...")

    original_fid = lcmodel_component_to_fid(
        basis,
        metabolite,
    )

    reference_fid, hlsvd_info = (
        model_reference_peak_hlsvd(
            original_fid,
            dwell_time=basis.dwell_time,
            hz_per_ppm=basis.hz_per_ppm,
            ppm_limits=(-0.2, 0.2),
            ppm_reference=4.65,
            n_singular_values=5,
            n_fit_points=8192,
        )
    )

    clean_fid = original_fid - reference_fid

    original_fids.append(original_fid)
    clean_fids.append(clean_fid)
    reference_fids.append(reference_fid)

    hlsvd_info_by_metabolite[metabolite] = hlsvd_info


original_fids = np.stack(
    original_fids,
    axis=0,
)

clean_fids = np.stack(
    clean_fids,
    axis=0,
)

reference_fids = np.stack(
    reference_fids,
    axis=0,
)


# ------------------------------------------------------------
# Transform all components to the frequency domain
# ------------------------------------------------------------
original_spectra = np.fft.fftshift(
    np.fft.fft(
        original_fids,
        axis=-1,
    ),
    axes=-1,
)

clean_spectra = np.fft.fftshift(
    np.fft.fft(
        clean_fids,
        axis=-1,
    ),
    axes=-1,
)

reference_spectra = np.fft.fftshift(
    np.fft.fft(
        reference_fids,
        axis=-1,
    ),
    axes=-1,
)

residual_spectra = (
    original_spectra
    - clean_spectra
)


# ------------------------------------------------------------
# Common ppm axis
# ------------------------------------------------------------
n_points = original_fids.shape[-1]
spectral_width_hz = basis.bandwidth

frequency_hz = np.linspace(
    -spectral_width_hz / 2
    + spectral_width_hz / (2 * n_points),
    spectral_width_hz / 2
    - spectral_width_hz / (2 * n_points),
    n_points,
)

ppm = (
    frequency_hz / basis.hz_per_ppm
    + 4.65
)


# ------------------------------------------------------------
# Raster plot: original vs cleaned vs residual
# ------------------------------------------------------------
n_spectra = basis.n_metabolites
n_columns = 4
n_rows = math.ceil(n_spectra / n_columns)

fig, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(16, 3 * n_rows),
    sharex=True,
)

axes = np.asarray(axes).ravel()


for ax, metabolite, original, cleaned, residual in zip(
    axes,
    basis.names,
    original_spectra,
    clean_spectra,
    residual_spectra,
):
    ax.plot(
        ppm,
        np.abs(original),
        label="Original",
    )

    ax.plot(
        ppm,
        np.abs(cleaned),
        linestyle="--",
        label="After HLSVD",
    )

    ax.plot(
        ppm,
        np.abs(residual),
        linestyle=":",
        label="Residual",
        alpha=0.8,
    )

    ax.set_title(metabolite)
    ax.set_xlim(4.5, 0.0)
    ax.grid(alpha=0.3)


for ax in axes[n_spectra:]:
    ax.axis("off")


handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper right",
)

fig.supxlabel("Chemical shift [ppm]")
fig.supylabel("Magnitude [a.u.]")
fig.suptitle(
    "LCModel basis components before and after HLSVD",
    fontsize=16,
)

plt.tight_layout(
    rect=(0, 0, 0.97, 0.97)
)

plt.show()

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Literal
import hashlib
import json
import platform
import subprocess
import sys

import h5py
import numpy as np


BASIS_LIBRARY_FORMAT = "walinet_lcmodel_basis_library"
BASIS_LIBRARY_FORMAT_VERSION = "1.0"


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def compute_sha256(path: str | Path) -> str:
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def get_git_commit(
    repository_path: str | Path | None = None,
) -> str:
    """
    Return current Git commit when available.
    Otherwise return 'unknown'.
    """
    cwd = (
        Path(repository_path)
        if repository_path is not None
        else Path.cwd()
    )

    try:
        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=cwd,
            check=True,
            capture_output=True,
            text=True,
        )

        return result.stdout.strip()

    except (
        FileNotFoundError,
        subprocess.CalledProcessError,
    ):
        return "unknown"


def make_source_id(
    source_path: str | Path,
    source_sha256: str,
) -> str:
    """
    Generate a readable, content-addressed source identifier.
    """
    source_path = Path(source_path)

    safe_stem = "".join(
        character
        if character.isalnum() or character in "-_"
        else "_"
        for character in source_path.stem
    )

    return f"{safe_stem}_{source_sha256[:12]}"


def write_array(
    group: h5py.Group,
    name: str,
    array: np.ndarray,
) -> h5py.Dataset:
    """
    Store an array with compression and integrity checking.
    """
    array = np.asarray(array)

    return group.create_dataset(
        name,
        data=array,
        compression="gzip",
        compression_opts=4,
        shuffle=True,
        fletcher32=True,
    )


def initialize_basis_library(
    h5: h5py.File,
    *,
    processing_git_commit: str,
) -> None:
    """
    Initialize root metadata when creating a new library.
    """
    h5.attrs["format"] = BASIS_LIBRARY_FORMAT
    h5.attrs["format_version"] = BASIS_LIBRARY_FORMAT_VERSION

    h5.attrs["created_utc"] = utc_now()
    h5.attrs["last_updated_utc"] = utc_now()

    h5.attrs["python_version"] = sys.version
    h5.attrs["platform"] = platform.platform()

    h5.attrs["numpy_version"] = np.__version__
    h5.attrs["h5py_version"] = h5py.__version__

    h5.attrs["processing_git_commit"] = (
        processing_git_commit
    )

    h5.require_group("sources")
    h5.require_group("components")
    h5.require_group("processing")


def validate_basis_library(h5: h5py.File) -> None:
    """
    Ensure an existing HDF5 file is a compatible WALINET basis library.
    """
    stored_format = h5.attrs.get("format")

    if stored_format != BASIS_LIBRARY_FORMAT:
        raise ValueError(
            "The selected HDF5 file is not a compatible "
            "WALINET LCModel basis library.\n"
            f"Found format: {stored_format!r}"
        )

    stored_version = str(
        h5.attrs.get("format_version")
    )

    if stored_version != BASIS_LIBRARY_FORMAT_VERSION:
        raise ValueError(
            "Unsupported basis-library format version.\n"
            f"Found: {stored_version}\n"
            f"Expected: {BASIS_LIBRARY_FORMAT_VERSION}"
        )


def register_source_basis(
    h5: h5py.File,
    *,
    source_basis_path: str | Path,
    basis: LCModelBasis,
) -> str:
    """
    Register and embed an original LCModel .basis file.

    Returns
    -------
    source_id:
        Stable identifier derived from filename and SHA-256.
    """
    source_basis_path = Path(
        source_basis_path
    ).resolve()

    if not source_basis_path.is_file():
        raise FileNotFoundError(
            f"Source basis file not found: "
            f"{source_basis_path}"
        )

    source_sha256 = compute_sha256(
        source_basis_path
    )

    source_id = make_source_id(
        source_basis_path,
        source_sha256,
    )

    sources_group = h5.require_group(
        "sources"
    )

    if source_id in sources_group:
        existing_group = sources_group[
            source_id
        ]

        existing_hash = existing_group.attrs[
            "sha256"
        ]

        if existing_hash != source_sha256:
            raise RuntimeError(
                "Source identifier collision detected."
            )

        return source_id

    source_group = sources_group.create_group(
        source_id
    )

    source_group.attrs["registered_utc"] = (
        utc_now()
    )

    source_group.attrs["filename"] = (
        source_basis_path.name
    )

    source_group.attrs["original_path"] = str(
        source_basis_path
    )

    source_group.attrs["sha256"] = (
        source_sha256
    )

    source_group.attrs["file_size_bytes"] = (
        source_basis_path.stat().st_size
    )

    source_group.attrs["sequence"] = (
        basis.sequence
    )

    source_group.attrs["echo_time"] = (
        basis.echo_time
    )

    source_group.attrs["dwell_time"] = (
        basis.dwell_time
    )

    source_group.attrs["bandwidth_hz"] = (
        basis.bandwidth
    )

    source_group.attrs["hz_per_ppm"] = (
        basis.hz_per_ppm
    )

    source_group.attrs["n_points"] = (
        basis.n_points
    )

    source_group.attrs["n_metabolites"] = (
        basis.n_metabolites
    )

    # Embed the complete original file.
    source_bytes = np.frombuffer(
        source_basis_path.read_bytes(),
        dtype=np.uint8,
    )

    write_array(
        source_group,
        "original_file",
        source_bytes,
    )

    return source_id


def save_hlsvd_information(
    component_group: h5py.Group,
    hlsvd_info: dict,
) -> None:
    """
    Store the full HLSVD result for one component.
    """
    if "hlsvd" in component_group:
        del component_group["hlsvd"]

    hlsvd_group = component_group.create_group(
        "hlsvd"
    )

    scalar_keys = (
        "n_fit_points",
        "n_singular_values_found",
    )

    for key in scalar_keys:
        if key in hlsvd_info:
            hlsvd_group.attrs[key] = int(
                hlsvd_info[key]
            )

    if "hankel_shape" in hlsvd_info:
        hlsvd_group.attrs["hankel_shape"] = (
            json.dumps(
                [
                    int(value)
                    for value in hlsvd_info[
                        "hankel_shape"
                    ]
                ]
            )
        )

    array_keys = (
        "singular_values",
        "frequencies_hz",
        "component_ppm",
        "damping_times",
        "amplitudes",
        "phases_deg",
        "selected",
        "selected_frequencies_hz",
        "selected_ppm",
        "selected_damping_times",
        "selected_amplitudes",
        "selected_phases_deg",
    )

    for key in array_keys:
        if key not in hlsvd_info:
            continue

        value = np.asarray(
            hlsvd_info[key]
        )

        write_array(
            hlsvd_group,
            key,
            value,
        )


def add_basis_component(
    h5: h5py.File,
    *,
    component_name: str,
    source_id: str,
    source_component_name: str,
    source_component_index: int,
    basis: LCModelBasis,
    original_fid: np.ndarray,
    clean_fid: np.ndarray,
    removed_reference_fid: np.ndarray,
    hlsvd_info: dict,
    duplicate_policy: Literal[
        "error",
        "skip",
        "replace",
    ] = "error",
) -> None:
    """
    Add one native basis component to the library.
    """
    components_group = h5.require_group(
        "components"
    )

    if component_name in components_group:
        if duplicate_policy == "error":
            raise ValueError(
                f"Component '{component_name}' already "
                f"exists in the library."
            )

        if duplicate_policy == "skip":
            print(
                f"[Skip] Component already exists: "
                f"{component_name}"
            )
            return

        if duplicate_policy == "replace":
            del components_group[
                component_name
            ]

        else:
            raise ValueError(
                f"Unknown duplicate policy: "
                f"{duplicate_policy}"
            )

    original_fid = np.asarray(
        original_fid,
        dtype=np.complex128,
    )

    clean_fid = np.asarray(
        clean_fid,
        dtype=np.complex128,
    )

    removed_reference_fid = np.asarray(
        removed_reference_fid,
        dtype=np.complex128,
    )

    if not (
        original_fid.shape
        == clean_fid.shape
        == removed_reference_fid.shape
    ):
        raise ValueError(
            f"Inconsistent FID shapes for "
            f"'{component_name}': "
            f"{original_fid.shape}, "
            f"{clean_fid.shape}, "
            f"{removed_reference_fid.shape}"
        )

    reconstruction_error = np.max(
        np.abs(
            clean_fid
            + removed_reference_fid
            - original_fid
        )
    )

    component_group = (
        components_group.create_group(
            component_name
        )
    )

    # Provenance
    component_group.attrs["created_utc"] = (
        utc_now()
    )

    component_group.attrs["source_id"] = (
        source_id
    )

    component_group.attrs[
        "source_component_name"
    ] = source_component_name

    component_group.attrs[
        "source_component_index"
    ] = int(source_component_index)

    component_group.attrs[
        "relative_scaling_preserved"
    ] = True

    component_group.attrs[
        "normalization_applied"
    ] = "none"

    component_group.attrs[
        "maximum_reconstruction_error"
    ] = float(reconstruction_error)

    # LCModel component metadata
    component_group.attrs["lcmodel_id"] = (
        basis.ids[source_component_index]
    )

    component_group.attrs[
        "lcmodel_concentration"
    ] = float(
        basis.concentrations[
            source_component_index
        ]
    )

    component_group.attrs["lcmodel_tramp"] = (
        float(
            basis.tramps[
                source_component_index
            ]
        )
    )

    component_group.attrs[
        "lcmodel_volume"
    ] = float(
        basis.volumes[
            source_component_index
        ]
    )

    component_group.attrs[
        "lcmodel_ishift"
    ] = int(
        basis.ishifts[
            source_component_index
        ]
    )

    # Native sampling metadata
    component_group.attrs["dwell_time"] = (
        basis.dwell_time
    )

    component_group.attrs["bandwidth_hz"] = (
        basis.bandwidth
    )

    component_group.attrs["hz_per_ppm"] = (
        basis.hz_per_ppm
    )

    component_group.attrs["n_points"] = (
        original_fid.size
    )

    # Processing metadata
    component_group.attrs[
        "reference_removal_method"
    ] = "HLSVDPROPY"

    component_group.attrs[
        "reference_ppm_min"
    ] = -0.2

    component_group.attrs[
        "reference_ppm_max"
    ] = 0.2

    component_group.attrs[
        "ppm_reference"
    ] = 4.65

    # Signals remain at native resolution.
    write_array(
        component_group,
        "original_fid",
        original_fid.astype(
            np.complex64
        ),
    )

    write_array(
        component_group,
        "clean_fid",
        clean_fid.astype(
            np.complex64
        ),
    )

    write_array(
        component_group,
        "removed_reference_fid",
        removed_reference_fid.astype(
            np.complex64
        ),
    )

    save_hlsvd_information(
        component_group,
        hlsvd_info,
    )

    print(
        f"[Added] {component_name} "
        f"<- {source_id}:{source_component_name}"
    )


def build_or_extend_basis_library(
    output_path: str | Path,
    *,
    source_basis_path: str | Path,
    basis: LCModelBasis,
    original_fids: np.ndarray,
    clean_fids: np.ndarray,
    reference_fids: np.ndarray,
    hlsvd_info_by_metabolite: dict,
    duplicate_policy: Literal[
        "error",
        "skip",
        "replace",
    ] = "error",
    processing_repository_path: str | Path | None = None,
) -> None:
    """
    Create or extend a native WALINET basis library.

    The original LCModel .basis file is embedded in the HDF5 file.
    Each metabolite is stored as an independent component group.
    """
    output_path = Path(output_path)

    expected_shape = (
        basis.n_metabolites,
        basis.n_points,
    )

    for name, array in (
        ("original_fids", original_fids),
        ("clean_fids", clean_fids),
        ("reference_fids", reference_fids),
    ):
        if array.shape != expected_shape:
            raise ValueError(
                f"{name} has shape {array.shape}; "
                f"expected {expected_shape}."
            )

    missing_info = [
        metabolite
        for metabolite in basis.names
        if metabolite
        not in hlsvd_info_by_metabolite
    ]

    if missing_info:
        raise ValueError(
            "Missing HLSVD metadata for: "
            + ", ".join(missing_info)
        )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    file_exists = output_path.exists()

    processing_git_commit = get_git_commit(
        processing_repository_path
    )

    with h5py.File(
        output_path,
        "a",
    ) as h5:
        if not file_exists:
            initialize_basis_library(
                h5,
                processing_git_commit=(
                    processing_git_commit
                ),
            )
        else:
            validate_basis_library(h5)

        source_id = register_source_basis(
            h5,
            source_basis_path=source_basis_path,
            basis=basis,
        )

        for index, metabolite in enumerate(
            basis.names
        ):
            add_basis_component(
                h5,
                component_name=metabolite,
                source_id=source_id,
                source_component_name=metabolite,
                source_component_index=index,
                basis=basis,
                original_fid=original_fids[index],
                clean_fid=clean_fids[index],
                removed_reference_fid=(
                    reference_fids[index]
                ),
                hlsvd_info=(
                    hlsvd_info_by_metabolite[
                        metabolite
                    ]
                ),
                duplicate_policy=duplicate_policy,
            )

        h5.attrs["last_updated_utc"] = (
            utc_now()
        )

        # Flush all pending writes before closing.
        h5.flush()

    print()
    print(
        f"Basis library saved: "
        f"{output_path.resolve()}"
    )

In [ ]:
from pathlib import Path


source_basis_path = Path(
    "LCModelBasis/raw/7T.basis"
)

output_library_path = Path(
    "LCModelBasis/processed/"
    "walinet_7T_native_basis_v1.h5"
)


build_or_extend_basis_library(
    output_library_path,
    source_basis_path=source_basis_path,
    basis=basis,
    original_fids=original_fids,
    clean_fids=clean_fids,
    reference_fids=reference_fids,
    hlsvd_info_by_metabolite=(
        hlsvd_info_by_metabolite
    ),
    duplicate_policy="error",
    processing_repository_path=".",
)

In [ ]:
def inspect_basis_library(
    path: str | Path,
) -> None:
    path = Path(path)

    with h5py.File(path, "r") as h5:
        validate_basis_library(h5)

        print("File:", path)
        print(
            "Format version:",
            h5.attrs["format_version"],
        )

        print("\nSources:")

        for source_id, source in h5[
            "sources"
        ].items():
            print(
                f"  {source_id}"
            )
            print(
                f"    file: "
                f"{source.attrs['filename']}"
            )
            print(
                f"    sha256: "
                f"{source.attrs['sha256']}"
            )

        print("\nComponents:")

        for name, component in h5[
            "components"
        ].items():
            print(
                f"  {name:12s} "
                f"<- "
                f"{component.attrs['source_id']}"
            )


inspect_basis_library(
    "LCModelBasis/processed/walinet_7T_native_basis_v1.h5"
)

In [ ]:
from pathlib import Path
import math

import h5py
import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Library laden
# ------------------------------------------------------------
library_path = Path(
    "LCModelBasis/processed/"
    "walinet_7T_native_basis_v1.h5"
)

with h5py.File(library_path, "r") as h5:
    component_group = h5["components"]

    names = list(component_group.keys())

    original_fids = np.stack(
        [
            component_group[name]["original_fid"][...]
            for name in names
        ],
        axis=0,
    )

    clean_fids = np.stack(
        [
            component_group[name]["clean_fid"][...]
            for name in names
        ],
        axis=0,
    )

    dwell_time = float(
        component_group[names[0]].attrs["dwell_time"]
    )

    hz_per_ppm = float(
        component_group[names[0]].attrs["hz_per_ppm"]
    )

    n_points = int(
        component_group[names[0]].attrs["n_points"]
    )


# ------------------------------------------------------------
# Konsistenzchecks
# ------------------------------------------------------------
if original_fids.shape != clean_fids.shape:
    raise ValueError(
        "Original and cleaned basis arrays have different shapes."
    )

if original_fids.shape[1] != n_points:
    raise ValueError(
        f"Stored n_points={n_points}, "
        f"but arrays contain {original_fids.shape[1]} points."
    )

print("Components:", names)
print("Original shape:", original_fids.shape)
print("Clean shape:", clean_fids.shape)
print("Dwell time:", dwell_time)
print("HZPPPM:", hz_per_ppm)


# ------------------------------------------------------------
# In den Frequenzraum
# ------------------------------------------------------------
original_spectra = np.fft.fftshift(
    np.fft.fft(
        original_fids,
        axis=-1,
    ),
    axes=-1,
)

clean_spectra = np.fft.fftshift(
    np.fft.fft(
        clean_fids,
        axis=-1,
    ),
    axes=-1,
)


# ------------------------------------------------------------
# Gemeinsame ppm-Achse
# ------------------------------------------------------------
spectral_width_hz = 1.0 / dwell_time

frequency_hz = np.linspace(
    -spectral_width_hz / 2
    + spectral_width_hz / (2 * n_points),
    spectral_width_hz / 2
    - spectral_width_hz / (2 * n_points),
    n_points,
)

ppm_reference = 4.65

ppm = (
    frequency_hz / hz_per_ppm
    + ppm_reference
)


# ------------------------------------------------------------
# Rasterplot
# ------------------------------------------------------------
n_components = len(names)
n_columns = 4
n_rows = math.ceil(n_components / n_columns)

fig, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(16, 3 * n_rows),
    sharex=True,
)

axes = np.asarray(axes).ravel()


for ax, name, original, cleaned in zip(
    axes,
    names,
    original_spectra,
    clean_spectra,
):
    ax.plot(
        ppm,
        np.abs(original),
        label="Original",
    )

    ax.plot(
        ppm,
        np.abs(cleaned),
        linestyle="--",
        label="Cleaned",
    )

    ax.set_title(name)
    ax.set_xlim(4.5, 0.0)
    ax.grid(alpha=0.3)


for ax in axes[n_components:]:
    ax.axis("off")


handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper right",
)

fig.supxlabel("Chemical shift [ppm]")
fig.supylabel("Magnitude [a.u.]")
fig.suptitle(
    "Original and cleaned basis components loaded from HDF5",
    fontsize=16,
)

plt.tight_layout(
    rect=(0, 0, 0.97, 0.97)
)

plt.show()

In [ ]:
from pathlib import Path

import h5py


library_path = Path(
    "LCModelBasis/processed/"
    "walinet_7T_native_basis_v1.h5"
)

metabolite = "Tau"


with h5py.File(library_path, "r") as h5:

    component = h5["components"][metabolite]

    source_id = component.attrs["source_id"]
    source = h5["sources"][source_id]

    print("=" * 70)
    print(f"Metabolite: {metabolite}")
    print("=" * 70)

    print("\nSource")
    print("------")
    print("Source ID:      ", source_id)
    print("Filename:       ", source.attrs["filename"])
    print("Original path:  ", source.attrs["original_path"])
    print("SHA256:         ", source.attrs["sha256"])

    print("\nAcquisition")
    print("-----------")
    print("Sequence:       ", source.attrs["sequence"])
    print("Echo time:      ", source.attrs["echo_time"])
    print("Dwell time:     ", source.attrs["dwell_time"])
    print("Bandwidth [Hz]: ", source.attrs["bandwidth_hz"])
    print("Hz / ppm:       ", source.attrs["hz_per_ppm"])
    print("Native points:  ", source.attrs["n_points"])

    print("\nLCModel metadata")
    print("----------------")
    print("ID:             ", component.attrs["lcmodel_id"])
    print("CONC:           ", component.attrs["lcmodel_concentration"])
    print("TRAMP:          ", component.attrs["lcmodel_tramp"])
    print("VOLUME:         ", component.attrs["lcmodel_volume"])
    print("ISHIFT:         ", component.attrs["lcmodel_ishift"])

    print("\nProcessing")
    print("----------")
    print("Reference removal:",
          component.attrs["reference_removal_method"])
    print("ppm reference:    ",
          component.attrs["ppm_reference"])
    print("ppm limits:       ",
          component.attrs["reference_ppm_min"],
          component.attrs["reference_ppm_max"])
    print("Reconstruction error:",
          component.attrs["maximum_reconstruction_error"])

In [ ]:
from pathlib import Path
import sys


project_root = Path("..").resolve()

src_path = project_root / "src"

if not src_path.exists():
    raise FileNotFoundError(
        f"Could not find src directory: {src_path}"
    )

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from walinet.training_data.lcmodel_basis.parser import (
    load_lcmodel_basis,
)

basis = load_lcmodel_basis(
    "LCModelBasis/raw/7T.basis",
)

print(f"Metabolites : {basis.n_metabolites}")
print(f"Time points : {basis.n_points}")
print(f"Dwell time  : {basis.dwell_time:.9e} s")
print(f"Bandwidth   : {basis.bandwidth:.2f} Hz")
print(f"Hz / ppm    : {basis.hz_per_ppm:.3f}")

In [ ]:
from walinet.training_data.lcmodel_basis.hlsvd import (
    process_lcmodel_basis,
)

processed_basis = process_lcmodel_basis(
    basis,
    ppm_limits=(-0.2, 0.2),
    ppm_reference=4.65,
    n_singular_values=5,
    n_fit_points=8192,
)

print()
print("Finished.")
print(
    "Maximum reconstruction error:",
    (
        abs(
            processed_basis.original_fids
            - (
                processed_basis.clean_fids
                + processed_basis.reference_fids
            )
        )
    ).max(),
)

In [ ]:
from walinet.training_data.lcmodel_basis.plotting import (
    plot_basis_before_after_grid,
)

plot_basis_before_after_grid(
    basis,
    processed_basis,
    ppm_limits=(7.5, 0),
)

In [ ]:
from pathlib import Path

from walinet.training_data.lcmodel_basis.library import (
    build_or_extend_basis_library,
)


basis_path = (
    project_root
    / "MetabModes/LCModelBasis"
    / "raw/7T.basis"
)

output_library_path = (
    project_root
    / "MetabModes/LCModelBasis"
    / "processed"
    / "walinet_7T_native_basis_v1.h5"
)


build_or_extend_basis_library(
    output_library_path,
    source_basis_path=basis_path,
    basis=basis,
    processed_basis=processed_basis,
    duplicate_policy="error",
    processing_repository_path=project_root,
)

In [ ]:
from walinet.training_data.lcmodel_basis.plotting import (
    plot_basis_library_consistency,
)


plot_basis_library_consistency(
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/MetabModes/LCModelBasis/processed/walinet_7T_native_basis_v1.h5",
    ppm_limits=(7.5, 0.0),
    n_columns=4,
)